# Turan — обучение детектора орфо-ошибок (KazRoBERTa) на GPU

Дообучаем `kz-transformers/kaz-roberta-conversational` как **token-classification детектор**
(по каждому подслову: KEEP / ERROR) на синтетическом датасете казахских орфо-ошибок.

**Перед запуском:** Runtime → Change runtime type → **GPU (T4)**.
Залей в Google Drive папку `turan_kk_spell/` с файлами `train.jsonl.gz`, `val.jsonl.gz`, `test.jsonl.gz`.

In [ ]:
!nvidia-smi -L
!pip install -q -U transformers datasets accelerate

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
# ⬇️ путь к папке с залитыми *.jsonl.gz
DATA_DIR = '/content/drive/MyDrive/turan_kk_spell'
OUT_DIR  = f'{DATA_DIR}/kazakh_detector'   # сюда сохранится обученная модель

In [ ]:
import numpy as np
from datasets import load_dataset
from transformers import (AutoTokenizer, AutoModelForTokenClassification,
                          DataCollatorForTokenClassification, Trainer, TrainingArguments)

MODEL_ID = "kz-transformers/kaz-roberta-conversational"
LABELS   = ["KEEP", "ERROR"]
MAX_LEN  = 128

In [ ]:
ds = load_dataset("json", data_files={
    "train":      f"{DATA_DIR}/train.jsonl.gz",
    "validation": f"{DATA_DIR}/val.jsonl.gz",
    "test":       f"{DATA_DIR}/test.jsonl.gz",
})
print(ds)

In [ ]:
def to_words_tags(ex):
    cw, rw = ex["corrupted"].split(), ex["correct"].split()
    if len(cw) != len(rw):
        return {"words": cw, "tags": [0]*len(cw)}
    return {"words": cw, "tags": [int(a != b) for a, b in zip(cw, rw)]}

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
assert tokenizer.is_fast

def tokenize_align(batch):
    enc = tokenizer(batch["words"], is_split_into_words=True,
                    truncation=True, max_length=MAX_LEN)
    out = []
    for i, tags in enumerate(batch["tags"]):
        prev, lab = None, []
        for wid in enc.word_ids(batch_index=i):
            if wid is None:      lab.append(-100)
            elif wid != prev:    lab.append(tags[wid])
            else:                lab.append(-100)
            prev = wid
        out.append(lab)
    enc["labels"] = out
    return enc

cols = ds["train"].column_names
ds = ds.map(to_words_tags)
ds = ds.map(tokenize_align, batched=True, remove_columns=cols + ["words", "tags"])

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    m = labels != -100
    yt, yp = labels[m], preds[m]
    tp = int(((yp==1)&(yt==1)).sum()); fp = int(((yp==1)&(yt==0)).sum()); fn = int(((yp==0)&(yt==1)).sum())
    prec = tp/(tp+fp) if tp+fp else 0.0
    rec  = tp/(tp+fn) if tp+fn else 0.0
    f1   = 2*prec*rec/(prec+rec) if prec+rec else 0.0
    return {"error_precision":prec, "error_recall":rec, "error_f1":f1,
            "accuracy": float((yt==yp).mean())}

In [ ]:
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_ID, num_labels=2,
    id2label=dict(enumerate(LABELS)), label2id={l:i for i,l in enumerate(LABELS)})

args = TrainingArguments(
    output_dir=OUT_DIR,
    eval_strategy="epoch", save_strategy="epoch",
    learning_rate=2e-5, weight_decay=0.01,
    per_device_train_batch_size=64, per_device_eval_batch_size=128,
    num_train_epochs=2,           # для детектора 2 эпох с запасом (можно 1)
    fp16=True, logging_steps=200,
    load_best_model_at_end=True, metric_for_best_model="error_f1",
    report_to="none",
)
trainer = Trainer(
    model=model, args=args,
    train_dataset=ds["train"], eval_dataset=ds["validation"],
    processing_class=tokenizer,
    data_collator=DataCollatorForTokenClassification(tokenizer),
    compute_metrics=compute_metrics,
)
trainer.train()

In [ ]:
print("TEST:", {k: round(v,4) for k,v in trainer.evaluate(ds["test"]).items() if isinstance(v,float)})
trainer.save_model(OUT_DIR)
tokenizer.save_pretrained(OUT_DIR)
print("Модель сохранена в", OUT_DIR)

## Дальше
Скачай папку `kazakh_detector/` из Drive — это обученный детектор (формат HuggingFace).
Локально он подключается в `model_loader.TransformerModel`, а исправление найденных
ошибок делает модуль правил `ru→kk` + лексикон + морфоанализатор.